<a href="https://colab.research.google.com/github/joshcova/NLP_Workshop/blob/main/04_FinBERT_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Afternoon exercise: FinBERT on the Financial PhraseBank

In this afternoon's exercise we move from understanding BERT-type models in the abstract to actually *using* one. The model of the day is **FinBERT** (`ProsusAI/finbert`), a BERT model fine-tuned for sentiment classification on financial text.

The dataset is the **Financial PhraseBank** (Malo et al., 2014), a collection of sentences from financial news that were hand-labelled by domain experts as *negative*, *neutral* or *positive*. We have pre-selected 200 sentences from the part of the PhraseBank that FinBERT was **not** trained on, so the comparisons here are honest.

The guiding question is simple:

> *You are a researcher who just read that FinBERT exists. Can you actually use it — > and should you trust it?*

We will work through four tasks, each building on the last:

1. **Make a prediction** — get FinBERT running on a handful of sentences you pick yourself.
2. **Scale it up** — apply FinBERT to all 200 sentences and compare its predictions to the human labels.
3. **Compare to a generalist** — run the same sentences through a non-specialised sentiment model (`distilbert-base-uncased-finetuned-sst-2-english`) and see how they differ.
4. **Look at the disagreements** — read the sentences where FinBERT got it wrong and try to find a pattern.

At the end of each task there is a short **reflection prompt**. Jot down a sentence or two — we will collect these in the plenary.

## Setup

We need two libraries on top of the usual scientific Python stack:

- `transformers` — Hugging Face's interface for loading pre-trained models.
- `scikit-learn` — for classification metrics (accuracy, confusion matrix).

Colab already has `pandas`, so we only need to install `transformers`. `scikit-learn` is also pre-installed.

In [ ]:
!pip install -q transformers

In [ ]:
import pandas as pd
from transformers import pipeline

# Silence the HF progress bars so the notebook stays readable.
from transformers.utils import logging
logging.set_verbosity_error()

### Loading FinBERT

Hugging Face's `pipeline()` is the highest-level interface: it bundles the tokenizer and the model, and hides all the plumbing. You give it a string, it gives you a prediction.

The first time you run the cell below, it will download the model weights (~440 MB). This takes a minute on Colab, and then the model is cached for the rest of the session.

In [ ]:
# "text-classification" tells the pipeline what kind of head to put on top of the model.
# The model card is: https://huggingface.co/ProsusAI/finbert
finbert = pipeline("text-classification", model="ProsusAI/finbert")

---
## Task 1 — Make a prediction

Your first goal is to get a feel for what a classifier model actually *returns*. We will feed FinBERT a few sentences and inspect its output.

Start with the example we provide. Then **replace two of the three sentences with ones you come up with yourself** — try to pick at least one sentence that you think is ambiguous or tricky.

In [ ]:
example_sentences = [
    "Operating profit rose to EUR 13.1 mn from EUR 8.7 mn in the corresponding period."
,    "The company announced layoffs affecting 10% of its workforce."
,    "The board will meet next Thursday at the headquarters in Helsinki."
,]

for sentence in example_sentences:
    prediction = finbert(sentence)
    print(sentence)
    print("  ->", prediction)
    print()

Take a moment to look at what came back. Each prediction is a list containing a dictionary with two keys:

- `label` — the predicted class (here: `positive`, `negative`, or `neutral`).
- `score` — a number between 0 and 1, the model's confidence in that label.

By default, `pipeline()` returns only the *top* label. If we want to see the full distribution across all three classes, we can pass `top_k=None`:

In [ ]:
sentence = "The company's quarterly revenue slightly missed analyst expectations."
print(finbert(sentence, top_k=None))

**Reflection 1.** In one sentence: what does the `score` mean, and what does it **not** mean? (Hint: is a score of 0.95 the same as saying *'95% of similar sentences are positive'*?)

---
## Task 2 — Scale it up

Running a model on three sentences is cute. Running it on 200 and checking how often it agrees with the human labels is research.

We load the 200-sentence sample from the Financial PhraseBank. Each row has:

- `id` — a stable identifier (1..200),
- `sentence` — the text,
- `true_label` — the human gold label (`negative`, `neutral`, `positive`).

In [ ]:
URL = "https://raw.githubusercontent.com/joshcova/NLP_Workshop/refs/heads/main/data/financial_phrasebank_sample.csv"
df = pd.read_csv(URL)
print("Number of sentences:", len(df))
print("Label distribution:")
print(df["true_label"].value_counts())
df.head()

Now we need to run FinBERT on every sentence. We wrap the model call in a small helper function that returns just the label (we'll keep the score too, in a second column).

In [ ]:
def predict_finbert(text):
    """Return (label, score) for a single sentence."""
    result = finbert(text)[0]
    return result["label"], result["score"]

# Applying the function creates a column of (label, score) tuples.
# We then split it into two columns. This takes ~30-60 seconds on Colab CPU.
df[["finbert_label", "finbert_score"]] = df["sentence"].apply(
    predict_finbert
).apply(pd.Series)

df.head()

### Evaluating the predictions

We have two things to compare now: `true_label` (what the humans said) and `finbert_label` (what FinBERT said). The two most basic evaluation tools are:

- **Accuracy** — the share of sentences where the labels agree.
- **Confusion matrix** — a table showing *which* classes get confused for which.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

labels = ["negative", "neutral", "positive"]  # fix the order so the matrix is readable

accuracy = accuracy_score(df["true_label"], df["finbert_label"])
print(f"FinBERT accuracy: {accuracy:.3f}")

cm = confusion_matrix(df["true_label"], df["finbert_label"], labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels],
                     columns=[f"pred_{l}" for l in labels])
print("\nConfusion matrix (rows = truth, columns = FinBERT prediction):")
print(cm_df)

In [ ]:
# A per-class report gives us precision, recall, and F1 for each label.
print(classification_report(df["true_label"], df["finbert_label"], labels=labels))

**Reflection 2.** Look at the confusion matrix. Which class does FinBERT handle best? Which one does it confuse most often, and with *which* other class? Write one or two sentences.

---
## Task 3 — Compare to a generalist

FinBERT is a specialist. To see whether specialisation matters, we need a *generalist* to compare against. We will use `distilbert-base-uncased-finetuned-sst-2-english` — a DistilBERT fine-tuned on the Stanford Sentiment Treebank (SST-2), which consists of movie reviews.

One important difference: the generalist only knows two classes, `POSITIVE` and `NEGATIVE`. It has no concept of *neutral*. We will deal with this honestly by **restricting the comparison to the sentences the humans labelled as non-neutral** (positive or negative). This is where the two models can actually compete on the same terms.

In [ ]:
distilbert = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)

In [ ]:
def predict_distilbert(text):
    result = distilbert(text)[0]
    # Normalise labels to match FinBERT's lowercase convention.
    return result["label"].lower(), result["score"]

df[["distil_label", "distil_score"]] = df["sentence"].apply(
    predict_distilbert
).apply(pd.Series)

df.head()

Now we restrict to the non-neutral sentences (positive or negative in the human labels) and compare both models head-to-head.

In [ ]:
subset = df[df["true_label"].isin(["positive", "negative"])].copy()
print(f"Non-neutral sentences: {len(subset)}")

# Collapse FinBERT's 3-class output: we keep its positive/negative calls as-is, and treat
# any 'neutral' prediction as a miss (since the truth here is never 'neutral').
finbert_acc = (subset["finbert_label"] == subset["true_label"]).mean()
distil_acc = (subset["distil_label"] == subset["true_label"]).mean()

print(f"FinBERT   accuracy on non-neutral: {finbert_acc:.3f}")
print(f"DistilBERT accuracy on non-neutral: {distil_acc:.3f}")

In [ ]:
# Per-class breakdown so we can see where each model shines or fails.
for model_col, name in [("finbert_label", "FinBERT"), ("distil_label", "DistilBERT")]:
    print(f"\n=== {name} ===")
    print(classification_report(
        subset["true_label"], subset[model_col],
        labels=["negative", "positive"], zero_division=0,
    ))

**Reflection 3.** Where does domain specialisation actually pay off? Write one sentence. (Bonus: can you imagine a use case where the generalist would be the *better* choice?)

---
## Task 4 — Look at the disagreements

Metrics summarise, but they also hide. The really instructive cases are the ones where the models disagree — with the humans, and with each other. Reading those sentences is the single most important thing you can do to build an intuition for what a classifier model is doing.

We pull out two small sets of disagreements and read them.

In [ ]:
# (a) Sentences where FinBERT disagrees with the human label.
finbert_wrong = df[df["finbert_label"] != df["true_label"]].copy()
print(f"FinBERT disagrees with humans on {len(finbert_wrong)} of {len(df)} sentences.")

# Show 10 of them (sorted by FinBERT's confidence — the high-confidence errors are the
# most interesting).
finbert_wrong.sort_values("finbert_score", ascending=False).head(10)[
    ["sentence", "true_label", "finbert_label", "finbert_score"]
]

In [ ]:
# (b) Sentences where FinBERT and DistilBERT disagree with each other (on non-neutral truth).
model_disagree = subset[subset["finbert_label"] != subset["distil_label"]].copy()
print(f"The two models disagree on {len(model_disagree)} of {len(subset)} non-neutral sentences.")

model_disagree.head(10)[
    ["sentence", "true_label", "finbert_label", "distil_label"]
]

**Reflection 4.** Read the sentences in both tables above. Write **one sentence** describing a pattern you notice in FinBERT's mistakes. Possible angles:

- What kind of language trips it up (numbers, negations, hedging, forward-looking statements...)?
- Are the mistakes evenly distributed across classes, or clustered?
- Do FinBERT and DistilBERT fail in *similar* ways, or in *different* ways?

Bring this observation to the plenary — we'll build on it together.

---
## What's next

In the plenary we will:

1. Pool the patterns you found in Task 4 into a rough taxonomy of FinBERT's blind spots.
2. Construct sentences together that probe those blind spots, and run both models on them live.
3. Step back and ask what it would take to *validate* FinBERT for a real research paper.